Import  Libraries

In [30]:
# Additional imports for sentence segmentation (add to your existing imports)
import nltk
from nltk.tokenize import sent_tokenize
import pandas as pd
import json
from typing import List, Dict, Any
import logging  
import spacy
import re
import os
# Download NLTK data (run once)
try:
    nltk.data.find('tokenizers/punkt')
    print("NLTK punkt tokenizer already available")
except LookupError:
    print("Downloading NLTK punkt tokenizer...")
    nltk.download('punkt')
    print("Download complete")

print("All required libraries imported successfully!")

NLTK punkt tokenizer already available
All required libraries imported successfully!


In [31]:
# Configuration Cell - Define your global variables here
BASE_MODEL_NAME = "en_core_web_sm"  # or whatever spaCy model you're using
TARGET_BASE_LABELS = ["LULC", "PROCESS", "SURFACE_UNIT", "COORDINATES", "CHANGE", "RESEARCH_TERM"]

# Vocabulary file paths (update these paths to your actual files)
LULC_VOCAB_PATH = "LULC.csv"  # Update this path
PROCESS_VOCAB_PATH = "LCprocess.csv"  # Update this path
VOCAB_TERM_COLUMN = "term"  # Column name in your vocab CSV files

# Data file configuration  
CSV_FILE_PATH = 'extracted_data_full.csv'
TEXT_COLUMNS_TO_CONCATENATE = ['title', 'abstract', 'sections']

print("Configuration variables defined successfully!")
print(f"Target labels: {TARGET_BASE_LABELS}")
print(f"Base model: {BASE_MODEL_NAME}")

Configuration variables defined successfully!
Target labels: ['LULC', 'PROCESS', 'SURFACE_UNIT', 'COORDINATES', 'CHANGE', 'RESEARCH_TERM']
Base model: en_core_web_sm


In [32]:
# Your existing helper functions (keeping them as-is)
def load_vocabulary_from_csv(csv_path, term_column="term"):
    """Loads terms from a specified column in a CSV file."""
    global VOCAB_TERM_COLUMN
    effective_term_column = term_column
    if term_column == "term" and 'VOCAB_TERM_COLUMN' in globals():
        effective_term_column = VOCAB_TERM_COLUMN

    logging.info(f"Attempting to load vocabulary from: {csv_path} using term column: '{effective_term_column}'")
    if not os.path.exists(csv_path):
        logging.error(f"Vocabulary file not found at path: {csv_path}")
        return set()
    try:
        try:
            df_vocab = pd.read_csv(csv_path, encoding='utf-8')
        except UnicodeDecodeError:
            logging.warning(f"UTF-8 decoding failed for {csv_path}, trying 'latin1' encoding.")
            df_vocab = pd.read_csv(csv_path, encoding='latin1')

        if effective_term_column not in df_vocab.columns:
            logging.error(f"Error: Column '{effective_term_column}' not found in vocabulary file: {csv_path}")
            logging.error(f"Available columns: {df_vocab.columns.tolist()}")
            return set()

        vocab_set = set(df_vocab[effective_term_column].dropna().astype(str).str.lower().str.strip().unique())
        vocab_set.discard('')
        logging.info(f"Successfully loaded {len(vocab_set)} unique terms from {csv_path} (column: '{effective_term_column}')")
        return vocab_set
    except Exception as e:
        logging.error(f"Error loading vocabulary from {csv_path}: {e}", exc_info=True)
        return set()

def _generate_patterns_for_single_vocab(nlp, vocab_set, label):
    """Generates patterns (LOWER and LEMMA) for terms in a vocabulary set."""
    vocab_patterns = []
    has_lemmatizer = nlp.has_pipe("lemmatizer")
    if not has_lemmatizer:
         logging.warning(f"SpaCy model '{nlp.meta.get('name', 'Unknown Model')}' does not appear to have a lemmatizer pipe. Will skip generating LEMMA patterns for '{label}' vocabulary.")

    for term in sorted(list(vocab_set)):
        if not term or not isinstance(term, str):
             logging.warning(f"Skipping invalid term in '{label}' vocabulary: {term}")
             continue
        try:
            doc_term = nlp(term.lower())
            if len(doc_term) == 0:
                logging.warning(f"Skipped '{label}' term '{term}': SpaCy tokenization resulted in an empty Doc.")
                continue
            lower_pattern = [{"LOWER": token.lower_} for token in doc_term if token.text.strip()]
            if lower_pattern:
                vocab_patterns.append({"label": label, "pattern": lower_pattern})
            else:
                logging.warning(f"Skipped '{label}' term '{term}': Could not generate valid LOWER patterns after tokenization/stripping.")
            if has_lemmatizer:
                 lemma_pattern = [{"LEMMA": token.lemma_} for token in doc_term if token.text.strip() and token.lemma_ and token.lemma_ != "-"]
                 if lemma_pattern:
                     vocab_patterns.append({"label": label, "pattern": lemma_pattern})
                 else:
                     logging.warning(f"Skipped '{label}' term '{term}': Could not generate valid LEMMA patterns (check tokenization/lemmatization results).")
        except Exception as e:
             logging.warning(f"Error generating patterns for '{label}' term '{term}': {e}")
    logging.info(f"Generated {len(vocab_patterns)} total patterns (LOWER + LEMMA attempts) for '{label}' vocabulary.")
    return vocab_patterns

def generate_ruler_patterns(nlp_tokenizer_like, lulc_vocab, process_vocab, extra_labels=None):
    """Generates a list of SpaCy patterns from vocabulary and specific rules."""
    patterns = []
    global TARGET_BASE_LABELS

    logging.info("Generating patterns from vocabularies...")
    try:
        if "LULC" in TARGET_BASE_LABELS:
            patterns.extend(_generate_patterns_for_single_vocab(nlp_tokenizer_like, lulc_vocab, "LULC"))
        else:
            logging.warning("Skipping LULC vocab patterns: 'LULC' not in TARGET_BASE_LABELS.")
        if "PROCESS" in TARGET_BASE_LABELS:
            patterns.extend(_generate_patterns_for_single_vocab(nlp_tokenizer_like, process_vocab, "PROCESS"))
        else:
            logging.warning("Skipping PROCESS vocab patterns: 'PROCESS' not in TARGET_BASE_LABELS.")
    except Exception as e:
        logging.error(f"Error generating vocabulary patterns: {e}", exc_info=True)

    # Add all your rule-based patterns here (SURFACE_UNIT, COORDINATES, CHANGE, RESEARCH_TERM)
    # ... (keeping your existing pattern generation code)
    
    # SURFACE_UNIT patterns
    if "SURFACE_UNIT" in TARGET_BASE_LABELS:
        surface_unit_patterns = [
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}],
            [{"TEXT": {"REGEX": r"^\d+([\.,]\d+)?$"}}, {"LOWER": {"IN": ["ha", "hectare", "hectares"]}}],
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["acres", "acre"]}}],
            # Add more patterns as needed
        ]
        for pattern in surface_unit_patterns:
            patterns.append({"label": "SURFACE_UNIT", "pattern": pattern})
    
    # COORDINATES patterns
    if "COORDINATES" in TARGET_BASE_LABELS:
        coord_patterns = [
            [{"LIKE_NUM": True}, {"TEXT": {"IN": [",", ";"]}}, {"LIKE_NUM": True}],
            # Add more coordinate patterns as needed
        ]
        for pattern in coord_patterns:
            patterns.append({"label": "COORDINATES", "pattern": pattern})
    
    # CHANGE patterns
    if "CHANGE" in TARGET_BASE_LABELS:
        change_terms = ["increase", "decrease", "change", "growth", "decline"]
        patterns.append({"label": "CHANGE", "pattern": [{"LOWER": {"IN": change_terms}}]})
    
    logging.info(f"Generated a total of {len(patterns)} patterns for EntityRuler.")
    return patterns

def setup_nlp_pipeline_for_doccano(base_model_name, patterns_to_add, target_labels_for_ruler_rules):
    """Sets up a SpaCy NLP pipeline with custom entity ruler."""
    nlp_for_doccano = None
    try:
        logging.info(f"Loading base SpaCy model '{base_model_name}'")
        nlp_for_doccano = spacy.load(base_model_name)
        logging.info(f"SpaCy model '{base_model_name}' loaded. Default pipes: {nlp_for_doccano.pipe_names}")

        ruler_custom_patterns_filtered = [p for p in patterns_to_add if p.get('label') in target_labels_for_ruler_rules]
        
        ruler_pipe_name = "custom_entity_ruler"
        if ruler_pipe_name in nlp_for_doccano.pipe_names:
            nlp_for_doccano.remove_pipe(ruler_pipe_name)

        ruler_config = {"overwrite_ents": True}
        if "ner" in nlp_for_doccano.pipe_names:
            ruler = nlp_for_doccano.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, before="ner")
        else:
            ruler = nlp_for_doccano.add_pipe("entity_ruler", name=ruler_pipe_name, config=ruler_config, last=True)
        
        if ruler_custom_patterns_filtered:
            ruler.add_patterns(ruler_custom_patterns_filtered)
            logging.info(f"Custom EntityRuler configured with {len(ruler_custom_patterns_filtered)} patterns.")
        
        logging.info(f"Final pipeline: {nlp_for_doccano.pipe_names}")
        return nlp_for_doccano
    except Exception as e:
        logging.error(f"Failed to set up NLP pipeline: {e}", exc_info=True)
        return None

print("Helper functions defined successfully!")

Helper functions defined successfully!


In [33]:
# Setup your custom NLP pipeline
print("Setting up custom NLP pipeline...")

try:
    # Load base spaCy model for pattern generation
    nlp_base = spacy.load(BASE_MODEL_NAME)
    print(f"✓ Loaded base model: {BASE_MODEL_NAME}")
    
    # Load vocabularies (update paths in configuration cell)
    print("Loading vocabularies...")
    
    # For demo purposes, create empty vocabularies if files don't exist
    # Replace this with actual file loading when you have the files
    if os.path.exists(LULC_VOCAB_PATH):
        lulc_vocab = load_vocabulary_from_csv(LULC_VOCAB_PATH, VOCAB_TERM_COLUMN)
    else:
        print(f"Warning: LULC vocab file not found at {LULC_VOCAB_PATH}, using empty set")
        lulc_vocab = set()
    
    if os.path.exists(PROCESS_VOCAB_PATH):
        process_vocab = load_vocabulary_from_csv(PROCESS_VOCAB_PATH, VOCAB_TERM_COLUMN)
    else:
        print(f"Warning: Process vocab file not found at {PROCESS_VOCAB_PATH}, using empty set")
        process_vocab = set()
    
    print(f"✓ Loaded LULC vocab: {len(lulc_vocab)} terms")
    print(f"✓ Loaded Process vocab: {len(process_vocab)} terms")
    
    # Generate patterns
    print("Generating entity patterns...")
    patterns = generate_ruler_patterns(nlp_base, lulc_vocab, process_vocab)
    print(f"✓ Generated {len(patterns)} patterns")
    
    # Create custom NLP pipeline
    print("Creating custom NLP pipeline...")
    nlp_for_doccano = setup_nlp_pipeline_for_doccano(BASE_MODEL_NAME, patterns, TARGET_BASE_LABELS)
    
    if nlp_for_doccano:
        print("✓ Custom NLP pipeline created successfully!")
        print(f"Pipeline components: {nlp_for_doccano.pipe_names}")
        
        # Test the pipeline
        test_sentence = "The study area covers 500 hectares and shows land use change."
        doc = nlp_for_doccano(test_sentence)
        print(f"\nTest: '{test_sentence}'")
        print("Entities found:")
        for ent in doc.ents:
            print(f"  {ent.text} -> {ent.label_}")
    else:
        print("✗ Failed to create custom NLP pipeline")
        
except Exception as e:
    print(f"Error setting up pipeline: {e}")
    # Fallback to basic model
    nlp_for_doccano = spacy.load(BASE_MODEL_NAME)
    print("Using basic spaCy model as fallback")

Setting up custom NLP pipeline...
✓ Loaded base model: en_core_web_sm
Loading vocabularies...
✓ Loaded LULC vocab: 171 terms
✓ Loaded Process vocab: 19 terms
Generating entity patterns...
✓ Generated 385 patterns
Creating custom NLP pipeline...
✓ Custom NLP pipeline created successfully!
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'custom_entity_ruler', 'ner']

Test: 'The study area covers 500 hectares and shows land use change.'
Entities found:
  500 hectares -> SURFACE_UNIT
  change -> CHANGE


Token-Limited Sentence Segmentation and NER Processing Functions

In [34]:
def clean_text_for_segmentation(text: str) -> str:
    """Clean and preprocess text for better sentence segmentation"""
    if pd.isna(text):
        return ""
    
    # Convert to string and remove extra whitespace
    text = str(text).strip()
    
    # Remove extra spaces and newlines
    text = re.sub(r'\s+', ' ', text)
    
    # Remove problematic characters that might interfere with segmentation
    text = re.sub(r'[^\w\s.,!?;:()\-\'""]', ' ', text)
    
    return text

def count_tokens_simple(text: str) -> int:
    """Simple token counting - approximately 1 token per 4 characters for English"""
    return len(text) // 4  # Rough approximation

def count_tokens_with_tokenizer(text: str, tokenizer=None) -> int:
    """More accurate token counting using a specific tokenizer"""
    if tokenizer is None:
        # Simple approximation if no tokenizer provided
        return len(text.split())  # Word-based approximation
    else:
        # If you have a specific tokenizer (uncomment and modify as needed)
        # return len(tokenizer.encode(text))
        return len(text.split())  # Fallback to word count

def segment_text_into_token_limited_paragraphs(text: str, max_tokens: int = 1024, 
                                               min_sentence_length: int = 10,
                                               tokenizer=None) -> List[Dict[str, Any]]:
    """
    Segment text into paragraphs that don't exceed max_tokens.
    Each paragraph ends when adding another sentence would exceed the limit.
    
    Args:
        text: Input text to segment
        max_tokens: Maximum tokens per paragraph (default 1024)
        min_sentence_length: Minimum character length for sentences
        tokenizer: Optional tokenizer for accurate token counting
    
    Returns:
        List of dictionaries with paragraph info
    """
    if not text or text.strip() == "":
        return []
    
    # Clean text first
    cleaned_text = clean_text_for_segmentation(text)
    
    # Segment into sentences using NLTK
    sentences = sent_tokenize(cleaned_text)
    
    # Filter out very short sentences
    sentences = [s.strip() for s in sentences if len(s.strip()) >= min_sentence_length]
    
    if not sentences:
        return []
    
    paragraphs = []
    current_paragraph = []
    current_tokens = 0
    paragraph_id = 0
    
    for sentence_idx, sentence in enumerate(sentences):
        # Count tokens in this sentence
        sentence_tokens = count_tokens_with_tokenizer(sentence, tokenizer)
        
        # Check if adding this sentence would exceed the limit
        if current_tokens + sentence_tokens > max_tokens and current_paragraph:
            # Save current paragraph and start a new one
            paragraph_text = " ".join(current_paragraph)
            paragraphs.append({
                'paragraph_id': paragraph_id,
                'paragraph_text': paragraph_text,
                'sentence_count': len(current_paragraph),
                'total_tokens': current_tokens,
                'sentences': current_paragraph.copy(),
                'sentence_indices': list(range(sentence_idx - len(current_paragraph), sentence_idx))
            })
            
            # Start new paragraph
            paragraph_id += 1
            current_paragraph = [sentence]
            current_tokens = sentence_tokens
        else:
            # Add sentence to current paragraph
            current_paragraph.append(sentence)
            current_tokens += sentence_tokens
    
    # Don't forget the last paragraph
    if current_paragraph:
        paragraph_text = " ".join(current_paragraph)
        paragraphs.append({
            'paragraph_id': paragraph_id,
            'paragraph_text': paragraph_text,
            'sentence_count': len(current_paragraph),
            'total_tokens': current_tokens,
            'sentences': current_paragraph.copy(),
            'sentence_indices': list(range(len(sentences) - len(current_paragraph), len(sentences)))
        })
    
    return paragraphs

def apply_custom_ner_to_paragraph(nlp_pipeline, paragraph_text: str, target_labels: List[str] = None) -> List[Dict[str, Any]]:
    """Apply your custom NER model to a paragraph (token-limited text chunk)"""
    try:
        # Process paragraph with your custom NLP pipeline
        doc = nlp_pipeline(paragraph_text)
        
        entities = []
        for ent in doc.ents:
            # Filter by target labels if specified
            if target_labels and ent.label_ not in target_labels:
                continue
                
            entity_info = {
                'text': ent.text,
                'label': ent.label_,
                'start': ent.start_char,
                'end': ent.end_char,
                'start_token': ent.start,
                'end_token': ent.end,
                'confidence': getattr(ent, 'conf', None)
            }
            entities.append(entity_info)
        
        return entities
    except Exception as e:
        logging.error(f"Error processing paragraph with NER: {e}")
        return []

def process_document_with_token_limit(doc_id: str, full_text: str, nlp_pipeline, 
                                      max_tokens: int = 1024, target_labels: List[str] = None) -> List[Dict[str, Any]]:
    """Process a single document: segment into token-limited paragraphs and apply NER"""
    results = []
    
    # Segment text into token-limited paragraphs
    paragraphs = segment_text_into_token_limited_paragraphs(full_text, max_tokens)
    
    for paragraph_info in paragraphs:
        paragraph_text = paragraph_info['paragraph_text']
        
        # Apply NER to each paragraph
        entities = apply_custom_ner_to_paragraph(nlp_pipeline, paragraph_text, target_labels)
        
        # Store results
        result = {
            'doc_id': doc_id,
            'paragraph_id': paragraph_info['paragraph_id'],
            'paragraph_text': paragraph_text,
            'sentence_count': paragraph_info['sentence_count'],
            'total_tokens': paragraph_info['total_tokens'],
            'paragraph_length': len(paragraph_text),
            'num_entities': len(entities),
            'entities': entities,
            'individual_sentences': paragraph_info['sentences'],
            'sentence_indices': paragraph_info['sentence_indices']
        }
        results.append(result)
    
    return results

print("Token-limited paragraph segmentation and NER processing functions defined successfully!")

Token-limited paragraph segmentation and NER processing functions defined successfully!


Main Processing Function

In [35]:
def process_csv_with_token_limited_ner(csv_path: str, text_columns: List[str], nlp_pipeline, 
                                       max_tokens: int = 1024, target_labels: List[str] = None, 
                                       limit: int = None) -> pd.DataFrame:
    """
    Main function to process CSV file: load data, segment into token-limited paragraphs, and apply NER
    
    Args:
        csv_path: Path to your CSV file
        text_columns: List of column names to concatenate for text processing
        nlp_pipeline: Your custom SpaCy NLP pipeline
        max_tokens: Maximum tokens per paragraph (default 1024)
        target_labels: List of entity labels to extract (optional)
        limit: Limit number of documents to process (for testing)
    
    Returns:
        DataFrame with paragraph-level NER results
    """
    
    # Load data
    logging.info(f"Loading data from {csv_path}")
    try:
        df = pd.read_csv(csv_path)
        logging.info(f"Loaded {len(df)} rows from {csv_path}")
    except Exception as e:
        logging.error(f"Error loading CSV: {e}")
        return pd.DataFrame()
    
    # Concatenate specified text columns
    def concatenate_text_columns(row):
        texts = []
        for col in text_columns:
            if col in row and pd.notna(row[col]):
                cleaned_text = clean_text_for_segmentation(row[col])
                if cleaned_text:
                    texts.append(cleaned_text)
        return " ".join(texts)
    
    # Create combined text column
    df['combined_text'] = df.apply(concatenate_text_columns, axis=1)
    
    # Create doc_id if not exists
    if 'doc_id' not in df.columns:
        df['doc_id'] = df.index.astype(str)
    
    # Apply limit if specified
    if limit and limit > 0:
        df = df.head(limit)
        logging.info(f"Limited processing to {limit} documents")
    
    # Process each document
    all_results = []
    total_docs = len(df)
    
    logging.info(f"Processing documents with max {max_tokens} tokens per paragraph...")
    
    for idx, row in df.iterrows():
        doc_id = str(row['doc_id'])
        combined_text = row['combined_text']
        
        if not combined_text.strip():
            logging.warning(f"Empty text for doc_id: {doc_id}")
            continue
        
        # Process document with token limit
        doc_results = process_document_with_token_limit(
            doc_id, combined_text, nlp_pipeline, max_tokens, target_labels
        )
        all_results.extend(doc_results)
        
        # Progress indicator
        if (idx + 1) % 10 == 0:
            logging.info(f"Processed {idx + 1}/{total_docs} documents...")
    
    logging.info(f"Processing complete! Total paragraphs: {len(all_results)}")
    
    # Convert to DataFrame
    if all_results:
        results_df = pd.DataFrame(all_results)
        return results_df
    else:
        logging.warning("No results generated")
        return pd.DataFrame()

print("Main token-limited processing function defined successfully!")

Main token-limited processing function defined successfully!


Statistics and Analysis Functions

In [36]:
def generate_token_summary_statistics(results_df: pd.DataFrame):
    """Generate and display summary statistics for token-limited processing"""
    if results_df.empty:
        print("No results to summarize")
        return
    
    total_documents = results_df['doc_id'].nunique()
    total_paragraphs = len(results_df)
    total_entities = results_df['num_entities'].sum()
    paragraphs_with_entities = (results_df['num_entities'] > 0).sum()
    
    print("\n" + "="*60)
    print("TOKEN-LIMITED PARAGRAPH NER SUMMARY STATISTICS")
    print("="*60)
    print(f"Total documents processed: {total_documents}")
    print(f"Total paragraphs created: {total_paragraphs}")
    print(f"Average paragraphs per document: {total_paragraphs/total_documents:.1f}")
    print(f"Total entities found: {total_entities}")
    print(f"Paragraphs with entities: {paragraphs_with_entities} ({paragraphs_with_entities/total_paragraphs*100:.1f}%)")
    print(f"Average entities per paragraph: {total_entities/total_paragraphs:.2f}")
    
    # Token distribution
    token_stats = results_df['total_tokens'].describe()
    print(f"\nToken distribution per paragraph:")
    print(f"  Mean: {token_stats['mean']:.1f} tokens")
    print(f"  Median: {token_stats['50%']:.1f} tokens")
    print(f"  Max: {token_stats['max']:.0f} tokens")
    print(f"  Min: {token_stats['min']:.0f} tokens")
    
    # Sentence count distribution
    sentence_stats = results_df['sentence_count'].describe()
    print(f"\nSentence count per paragraph:")
    print(f"  Mean: {sentence_stats['mean']:.1f} sentences")
    print(f"  Median: {sentence_stats['50%']:.1f} sentences")
    print(f"  Max: {sentence_stats['max']:.0f} sentences")
    print(f"  Min: {sentence_stats['min']:.0f} sentences")
    
    # Entity type distribution
    if total_entities > 0:
        entity_counts = {}
        for _, row in results_df.iterrows():
            for entity in row['entities']:
                label = entity['label']
                entity_counts[label] = entity_counts.get(label, 0) + 1
        
        print(f"\nEntity type distribution:")
        sorted_entities = sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)
        for entity_type, count in sorted_entities:
            percentage = (count / total_entities) * 100
            print(f"  {entity_type}: {count} ({percentage:.1f}%)")

def analyze_json_results(json_results: List[Dict]):
    """Analyze and validate JSON results"""
    if not json_results:
        print("No JSON results to analyze")
        return
    
    print("JSON Results Analysis:")
    print("="*50)
    
    total_sentences = len(json_results)
    sentences_with_entities = sum(1 for entry in json_results if entry['entities'])
    total_entities = sum(len(entry['entities']) for entry in json_results)
    
    print(f"Total sentences: {total_sentences}")
    print(f"Sentences with entities: {sentences_with_entities} ({sentences_with_entities/total_sentences*100:.1f}%)")
    print(f"Total entities: {total_entities}")
    print(f"Average entities per sentence: {total_entities/total_sentences:.2f}")
    
    # Entity type distribution
    entity_counts = {}
    for entry in json_results:
        for entity in entry['entities']:
            label = entity['label']
            entity_counts[label] = entity_counts.get(label, 0) + 1
    
    if entity_counts:
        print(f"\nEntity type distribution:")
        sorted_entities = sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)
        for entity_type, count in sorted_entities:
            percentage = (count / total_entities) * 100
            print(f"  {entity_type}: {count} ({percentage:.1f}%)")
    
    # Show examples of sentences with most entities
    entries_with_entities = [entry for entry in json_results if entry['entities']]
    if entries_with_entities:
        entries_with_entities.sort(key=lambda x: len(x['entities']), reverse=True)
        
        print(f"\nTop 3 sentences with most entities:")
        for i, entry in enumerate(entries_with_entities[:3]):
            print(f"\n{i+1}. Article ID: {entry['article_id']}")
            print(f"   Sentence: {entry['original_sentence'][:100]}...")
            
            # Fixed: Create entity list outside f-string
            entity_list = [f"{e['text']}({e['label']})" for e in entry['entities']]
            print(f"   Entities ({len(entry['entities'])}): {entity_list}")
    
    # Validation: Check if entity positions are correct
    print(f"\nValidating entity positions (checking first 10 entries)...")
    validation_errors = 0
    
    for i, entry in enumerate(json_results[:10]):
        sentence = entry['original_sentence']
        for entity in entry['entities']:
            extracted_text = sentence[entity['start_char']:entity['end_char']]
            if extracted_text != entity['text']:
                print(f"ERROR in {entry['article_id']}: Expected '{entity['text']}', got '{extracted_text}'")
                validation_errors += 1
    
    if validation_errors == 0:
        print("✓ All checked entity positions are correct!")
    else:
        print(f"⚠ Found {validation_errors} position errors in the sample")

print("Statistics and analysis functions defined successfully!")

Statistics and analysis functions defined successfully!


Main Execution Cell

In [37]:
# Configuration - Update these paths and settings
CSV_FILE_PATH = 'extracted_data_full.csv'
TEXT_COLUMNS_TO_CONCATENATE = ['title', 'abstract', 'sections']

# Token limit configuration
MAX_TOKENS_PER_PARAGRAPH = 1024  # Adjust as needed (512, 1024, 2048, etc.)

# For testing, you can limit the number of documents
LIMIT_DOCS = None  # Set to a number like 50 for testing, None for all documents

# Target labels to extract (use your TARGET_BASE_LABELS or customize)
target_labels_to_extract = TARGET_BASE_LABELS if 'TARGET_BASE_LABELS' in globals() else None

# Output format options
SAVE_CSV = False  # Set to True if you also want CSV output
SAVE_JSON = True  # Set to True for JSON output (your preferred format)

print("Starting token-limited paragraph NER processing...")
print(f"CSV File: {CSV_FILE_PATH}")
print(f"Text columns: {TEXT_COLUMNS_TO_CONCATENATE}")
print(f"Max tokens per paragraph: {MAX_TOKENS_PER_PARAGRAPH}")
print(f"Target labels: {target_labels_to_extract}")
print(f"Document limit: {LIMIT_DOCS}")
print(f"Output JSON format: {SAVE_JSON}")
print(f"Output CSV format: {SAVE_CSV}")

# Process the data
try:
    # Make sure you have your NLP pipeline ready
    if 'nlp_for_doccano' in globals():
        nlp_pipeline = nlp_for_doccano
        print("Using custom nlp_for_doccano pipeline")
    else:
        print("Warning: Using basic spaCy model. Please ensure your custom NLP pipeline is loaded.")
        nlp_pipeline = spacy.load("en_core_web_sm")
    
    # Process CSV with token-limited paragraph NER
    results_df = process_csv_with_token_limited_ner(
        csv_path=CSV_FILE_PATH,
        text_columns=TEXT_COLUMNS_TO_CONCATENATE,
        nlp_pipeline=nlp_pipeline,
        max_tokens=MAX_TOKENS_PER_PARAGRAPH,
        target_labels=target_labels_to_extract,
        limit=LIMIT_DOCS
    )
    
    if not results_df.empty:
        print(f"\nProcessing completed successfully!")
        print(f"Generated {len(results_df)} paragraph records")
        
        # Generate summary statistics for paragraphs
        generate_token_summary_statistics(results_df)
        
        # Convert to JSON format
        if SAVE_JSON:
            print(f"\nConverting to JSON format...")
            print("Re-processing sentences for accurate entity positions...")
            
            json_results = convert_results_to_json_format_alternative(results_df)
            print(f"Generated {len(json_results)} sentence-level JSON entries")
            
            # Save JSON results
            json_output_file = f'ner_results_{MAX_TOKENS_PER_PARAGRAPH}tokens.json'
            if save_results_as_json(json_results, json_output_file):
                print(f"JSON results saved successfully!")
                
                # Display sample JSON output
                display_json_sample(json_results, num_samples=3)
                
                # Analyze JSON results
                analyze_json_results(json_results)
        
        # Also save CSV if requested
        if SAVE_CSV:
            output_file = f'token_limited_ner_results_{MAX_TOKENS_PER_PARAGRAPH}tokens.csv'
            flattened_df = flatten_token_results_for_export(results_df)
            flattened_df.to_csv(output_file, index=False)
            print(f"\nCSV results also saved to: {output_file}")
        
    else:
        print("No results generated. Please check your data and configuration.")
        
except Exception as e:
    logging.error(f"Error during processing: {e}")
    print(f"Error occurred: {e}")

Starting token-limited paragraph NER processing...
CSV File: extracted_data_full.csv
Text columns: ['title', 'abstract', 'sections']
Max tokens per paragraph: 1024
Target labels: ['LULC', 'PROCESS', 'SURFACE_UNIT', 'COORDINATES', 'CHANGE', 'RESEARCH_TERM']
Document limit: None
Output JSON format: True
Output CSV format: False
Using custom nlp_for_doccano pipeline

Processing completed successfully!
Generated 445 paragraph records

TOKEN-LIMITED PARAGRAPH NER SUMMARY STATISTICS
Total documents processed: 118
Total paragraphs created: 445
Average paragraphs per document: 3.8
Total entities found: 11035
Paragraphs with entities: 443 (99.6%)
Average entities per paragraph: 24.80

Token distribution per paragraph:
  Mean: 888.3 tokens
  Median: 1004.0 tokens
  Max: 1024 tokens
  Min: 33 tokens

Sentence count per paragraph:
  Mean: 30.5 sentences
  Median: 31.0 sentences
  Max: 57 sentences
  Min: 1 sentences

Entity type distribution:
  LULC: 5819 (52.7%)
  CHANGE: 3433 (31.1%)
  PROCESS: 

In [38]:
# Optional: Additional analysis and validation
if 'json_results' in globals() and json_results:
    
    # Example: Filter sentences by entity types
    def filter_sentences_by_entity_type(json_results, entity_type):
        """Filter sentences that contain a specific entity type"""
        filtered = []
        for entry in json_results:
            if any(entity['label'] == entity_type for entity in entry['entities']):
                filtered.append(entry)
        return filtered
    
    # Example: Get statistics for each entity type
    if 'TARGET_BASE_LABELS' in globals():
        for label in TARGET_BASE_LABELS:
            sentences_with_label = filter_sentences_by_entity_type(json_results, label)
            print(f"\nSentences containing {label} entities: {len(sentences_with_label)}")
            
            if sentences_with_label:
                # Show one example
                example = sentences_with_label[0]
                entities_of_type = [e for e in example['entities'] if e['label'] == label]
                print(f"Example: '{example['original_sentence'][:100]}...'")
                print(f"  {label} entities: {[e['text'] for e in entities_of_type]}")
    
    # Example: Save filtered results
    def save_filtered_json(json_results, entity_type, output_file):
        """Save sentences containing specific entity type"""
        filtered = filter_sentences_by_entity_type(json_results, entity_type)
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(filtered, f, ensure_ascii=False, indent=2)
        print(f"Saved {len(filtered)} sentences with {entity_type} entities to {output_file}")
    
    # Uncomment to save filtered results for specific entity types
    # save_filtered_json(json_results, 'LULC', 'lulc_sentences.json')
    # save_filtered_json(json_results, 'COORDINATES', 'coordinates_sentences.json')

print("All cells completed successfully! Check the generated JSON file for your results.")


Sentences containing LULC entities: 3266
Example: 'Land use and land cover change detection and prediction in Bhutan's high altitude city of Thimphu, u...'
  LULC entities: ['city', 'urban']

Sentences containing PROCESS entities: 968
Example: 'Land use and land cover change detection and prediction in Bhutan's high altitude city of Thimphu, u...'
  PROCESS entities: ['urbanization']

Sentences containing SURFACE_UNIT entities: 129
Example: 'It is also defined as urban expansion, as the process of concentrating the population into cities, o...'
  SURFACE_UNIT entities: ['billion hectares', 'billion hectares', 'million hectares']

Sentences containing COORDINATES entities: 321
Example: 'Training samples used to create the classification model were collected through visual interpretatio...'
  COORDINATES entities: ['2000, 2009']

Sentences containing CHANGE entities: 2491
Example: 'Land use and land cover change detection and prediction in Bhutan's high altitude city of Thimphu, u...'
 